# Séance 2 — Ingestion incrémentale vers la couche Bronze
*Session 2 — Incremental ingestion to the Bronze layer*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/02_ingestion_bronze.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ○ démonstration |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | — hors parcours |

**À la fin de cette séance, vous saurez :**

1. expliquer l'architecture en médaillon (Bronze, Silver, Gold) et le rôle de chaque couche ;
2. écrire une ingestion **idempotente** à l'aide d'un registre des fichiers traités ;
3. ajouter des métadonnées techniques de traçabilité (fichier source, lot, horodatage) ;
4. lire une table partitionnée avec Spark et avec pandas.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

## 1. Le principe

La couche **Bronze** est une copie **fidèle** des fichiers reçus : on ne corrige rien, on ajoute seulement des métadonnées
(`_fichier_source`, `_lot_id`, `_horodatage_ingestion`, `_date_partition`). Si un nettoyage s'avère faux plus tard, on peut
toujours rejouer à partir de Bronze.

L'ingestion est **incrémentale** et **idempotente** : un registre (`_registre_ingestion.json`) mémorise les fichiers déjà traités;
relancer le traitement n'ajoute jamais de doublons. Sur Databricks, *Auto Loader* joue ce rôle avec des points de contrôle.

In [ ]:
cfg = preparer("ingerer")
registre = lire_json(cfg.chemin("bronze") / "_registre_ingestion.json")
print(len(registre["fichiers"]), "fichiers déjà ingérés; derniers :", registre["fichiers"][-3:])
bronze = table("bronze", "transactions")
print(bronze.shape)
bronze[["transaction_id", "montant", "_fichier_source", "_lot_id", "_horodatage_ingestion", "_date_partition"]].head()

## 2. La même table vue par Spark

In [ ]:
from fraude360.utils.spark import obtenir_spark, lire_table
spark = obtenir_spark(cfg)
sdf = lire_table(spark, cfg.chemin_table("bronze", "transactions"), cfg)
sdf.printSchema()
sdf.groupBy("_date_partition").count().orderBy("_date_partition").show(5)

### Exercice 2.1 — 🟣 Gestion + Informatique

Écrivez `fichiers_a_ingerer(disponibles, deja_ingeres)` qui renvoie, **triée**, la liste des noms de fichiers
disponibles qui ne figurent pas dans le registre. Elle doit être idempotente : l'appliquer deux fois de suite (en mettant le
registre à jour entre les deux) renvoie une liste vide la deuxième fois.

In [ ]:
# VOTRE CODE ICI
def fichiers_a_ingerer(disponibles, deja_ingeres):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(fichiers_a_ingerer(["b.csv", "a.csv", "c.csv"], ["b.csv"]) == ["a.csv", "c.csv"])
_reg = []
_reg += fichiers_a_ingerer(["x.csv", "y.csv"], _reg)
verifier(fichiers_a_ingerer(["x.csv", "y.csv"], _reg) == [], "Deuxième passage : rien de nouveau")
_dispo = [f.name for f in cfg.chemin("atterrissage").glob("*.csv")]
verifier(fichiers_a_ingerer(_dispo, registre["fichiers"]) == [], "Tous les fichiers du cours sont déjà ingérés")

### Exercice 2.2 — 🔷 Informatique — approfondissement

Écrivez `ajouter_metadonnees(df, nom_fichier, lot_id)` qui renvoie une **copie** de `df` avec les colonnes
`_fichier_source`, `_lot_id`, `_horodatage_ingestion` (maintenant, `pd.Timestamp.now()`) et `_date_partition` — la date
`AAAA-MM-JJ` extraite du nom `transactions_AAAA-MM-JJ.csv`.

In [ ]:
# VOTRE CODE ICI
def ajouter_metadonnees(df, nom_fichier, lot_id):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_t = pd.DataFrame({"transaction_id": ["T1", "T2"]})
_r = ajouter_metadonnees(_t, "transactions_2026-01-05.csv", "lot-007")
verifier(set(["_fichier_source", "_lot_id", "_horodatage_ingestion", "_date_partition"]) <= set(_r.columns))
verifier((_r["_date_partition"] == "2026-01-05").all() and "_lot_id" not in _t.columns, "Date extraite du nom, df d'origine intact")

### Exercice 2.3 — 🟢 Tous les parcours

Bronze est-elle une copie **fidèle** ? Calculez `n_atterrissage`, le nombre total de lignes dans les fichiers CSV
d'atterrissage, et `n_bronze`, le nombre de lignes de la table Bronze.

In [ ]:
# VOTRE CODE ICI
n_atterrissage = ...
n_bronze = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(n_atterrissage == n_bronze, "Bronze doit contenir exactement les lignes reçues (doublons compris)")

**❓ Question (🟢 Tous les parcours) —** Pourquoi ne pas supprimer les doublons dès la couche Bronze ?

*Votre réponse ici.*

## Bilan

- Bronze = copie fidèle + métadonnées : traçabilité et possibilité de rejouer.
- Un registre rend l'ingestion idempotente (pas de doublons si on relance).
- Le même code tourne en Spark local et sur Databricks (Auto Loader, Delta Lake).

**Pour aller plus loin :**

- `src/fraude360/ingestion/bronze.py` et `infra/databricks/` (job Databricks Workflows).

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*